# DATA304 · ColBERT 5와 혼합 오답으로 Cross-Encoder 추가 학습

Gold Cross **epoch 2**의 모델과 AdamW 상태를 불러와 **epoch 3과 4**를 학습한다. 두 에포크 모두 학습률은 **5e-6**이다. 쿼리마다 고정된 ColBERT **epoch 5**가 높게 평가한 오답 4개와 기존 Gold sampler의 오답 4개를 정답 상품과 비교한다.

SageMaker 프로젝트 폴더에서 기존 학습 커널로 **Run All**을 실행한다. 첫 실행은 Gold epoch 2에서 시작한다. 같은 노트북을 다시 실행하면 이 혼합 실험에서 완전히 저장한 에포크 다음부터 이어가며, 중간에 멈춘 에포크는 처음부터 다시 학습한다. 진행 로그의 전체 경로는 설정 셀에서 출력한다.


In [ ]:
from collections import defaultdict
from contextlib import nullcontext
from datetime import datetime
from functools import lru_cache
from itertools import chain, islice
from pathlib import Path
from tempfile import TemporaryDirectory
import copy
import hashlib
import json
import os
import random

import faiss
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModel, AutoModelForSequenceClassification, AutoTokenizer

# SageMaker의 기존 프로젝트 폴더다. 학습에 쓴 커널에서 위에서 아래로 실행한다.
PROJECT_DIR = Path('/home/ec2-user/SageMaker/20262R0136DATA30400')
RUN_DIR = PROJECT_DIR / 'data304_outputs'
CROSS_RESUME = RUN_DIR / 'checkpoints' / 'cross_gold_last.pt'
COLBERT_EPOCH = 5
BASE_CROSS_EPOCH = 2
TARGET_EPOCHS = (3, 4)
CROSS_LR = 5e-6
HARD_COUNT = 4
ORIGINAL_COUNT = 4
NEGATIVE_COUNT = HARD_COUNT + ORIGINAL_COUNT
VAL_LIMIT = 350  # split.json의 첫 350건. None이면 저장된 VAL 전체.

# 동일한 설정으로 TRAIN 오답과 VAL 후보를 한 번 검색해 파일로 저장한다.
ANN_PER_TOKEN = 1024
ANN_POOL = 900
CANDIDATE_TOPK = 100
MINING_CHUNK_SIZE = 50  # 이 쿼리 수마다 후보를 저장한다. 중단 후 완료된 묶음부터 재사용한다.

CATALOG_PATH = PROJECT_DIR / 'catalog.json'
GOLD_PATH = PROJECT_DIR / 'train_gold.json'
SPLIT_PATH = RUN_DIR / 'split.json'
# 이전 실험의 후보는 읽기 전용이다. 새로 만드는 모든 파일은 별도 폴더에 저장한다.
PREVIOUS_CACHE_ROOT = RUN_DIR / 'cross_hard_negative' / 'candidate_cache'
CROSS_ROOT = RUN_DIR / 'cross_hard4_mixed4_lr5e-6'
MIXED_RESUME_NAME = 'cross_mixed_last.pt'
MIXED_WEIGHTS_NAME = 'cross_mixed.pt'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16 = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
faiss.omp_set_num_threads(min(8, os.cpu_count() or 4))
if DEVICE.type == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
print('Cross 재개 파일', CROSS_RESUME, '| 고정 ColBERT epoch', COLBERT_EPOCH, '| 장치', DEVICE, flush=True)

## 체크포인트와 Gold 분할

| 용도 | `data304_outputs/` 아래에서 읽는 경로 |
|---|---|
| 고정 검색기 | `additional_training/epoch_05/checkpoints/colbert_gold.pt` |
| Cross 학습 시작점 | `checkpoints/cross_gold_last.pt`의 epoch 2 |
| Gold 쿼리 분할 | `split.json` |

상품 순서, 입력 길이와 배치 크기는 기존 실행에서 복원한다. 검색기 가중치·검색 설정·데이터가 일치하면 이전 상품 인덱스와 후보 파일을 읽을 수 있다. Cross 모델, optimizer, 손실표, 검증 결과와 로그는 모두 새 실험 경로에서 관리한다.


In [ ]:
# 기존 프로젝트의 UTF-8 JSON을 읽는다.
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8-sig'))


def write_json(path, value):
    # 완성된 설정을 나중에 점수표와 대조할 수 있도록 사람이 읽을 수 있는 형식으로 쓴다.
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


@lru_cache(maxsize=32)
def file_hash(path, size, modified_ns):
    # 모델과 입력이 같을 때만 캐시를 공유한다. 같은 실행에서 큰 파일을 매번 다시 읽지 않는다.
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


def sha256(path):
    # 파일을 교체했으면 크기 또는 수정 시각이 바뀌므로 새 해시를 얻는다.
    stat = path.stat()
    return file_hash(path, stat.st_size, stat.st_mtime_ns)


def signature(value):
    # 파일 내용과 설정이 일치하는 후보 캐시·실험을 찾기 위한 해시를 만든다.
    data = json.dumps(value, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(data).hexdigest()


def amp_context():
    # 원래 실행처럼 지원되는 CUDA에서는 bfloat16으로 순전파한다.
    return torch.autocast('cuda', dtype=torch.bfloat16) if USE_BF16 else nullcontext()


def load_pt(path):
    # CPU 메모리 매핑으로 저장한 모델과 AdamW에서 필요한 텐서부터 읽는다.
    return torch.load(path, map_location='cpu', mmap=True, weights_only=True)


def metadata_text(item):
    # 학습 때 사용한 네 필드의 이름, 순서, [SEP]를 그대로 지켜 임베딩 입력을 맞춘다.
    fields = [
        ('title', 'Product'),
        ('store', 'Store'),
        ('features', 'Features'),
        ('description', 'Description'),
    ]
    parts = [f'{label}: {item[key].strip()}' for key, label in fields if item[key].strip()]
    return ' [SEP] '.join(parts) or 'Product metadata not available'


def gold_query(row):
    # 리뷰 다섯 개는 원래 순서를 지킨다. Gold 카테고리 경로는 기존 방식의 오답을 뽑을 때 사용한다.
    if len(row['reviews']) != 5:
        raise ValueError(f'리뷰가 5개가 아닌 쿼리 {row["query_id"]}')
    return {
        'category_path': tuple(level.strip() for level in row['category_path'].split('>') if level.strip()),
        'query_id': row['query_id'],
        'item_id': row['item_id'],
        'reviews': [review['text'].strip() for review in row['reviews']],
    }


# config와 split을 기존 파일에서 읽으므로 Gold 상품 분할과 토큰 길이가 이전 실행과 일치한다.
CFG = {}
for path in (RUN_DIR / 'aspect_vocabulary_and_config.json', RUN_DIR / 'run_manifest.json'):
    if path.is_file():
        CFG.update(read_json(path)['config'])
if not CFG:
    raise FileNotFoundError(f'기존 실행의 config가 필요합니다. {RUN_DIR}')

RETRIEVER_DIR = RUN_DIR / 'additional_training' / f'epoch_{COLBERT_EPOCH:02d}'
RETRIEVER_WEIGHTS = RETRIEVER_DIR / 'checkpoints' / 'colbert_gold.pt'
for path in (CROSS_RESUME, RETRIEVER_WEIGHTS, CATALOG_PATH, GOLD_PATH, SPLIT_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'저장한 파일을 확인하세요. {path}')
ORIGINAL_MODE = CFG['negative_mode']  # 원래 Gold 실행은 mixed다.
if not (0 < HARD_COUNT < CANDIDATE_TOPK <= ANN_POOL and ORIGINAL_COUNT > 0 and ANN_PER_TOKEN > 0):
    raise ValueError('오답 수와 검색 후보 수를 확인하세요.')
if MINING_CHUNK_SIZE < 1 or (VAL_LIMIT is not None and VAL_LIMIT < 1):
    raise ValueError('후보 저장 단위와 지정한 VAL 쿼리 수는 양수여야 합니다.')

# 다른 실험의 epoch 3·4 파일을 Gold 시작점으로 잘못 지정하면 후보 검색 전에 알려준다.
base = load_pt(CROSS_RESUME)
if (
    not {'epoch', 'model_state_dict', 'optimizer_state_dict'} <= base.keys()
    or int(base['epoch']) != BASE_CROSS_EPOCH
    or not base['optimizer_state_dict']['state']
):
    raise ValueError(f'모델과 AdamW 상태를 포함한 Gold Cross epoch 2가 필요합니다. {CROSS_RESUME}')
gold_lrs = [float(group['lr']) for group in base['optimizer_state_dict']['param_groups']]
del base

catalog = read_json(CATALOG_PATH)
ITEM_IDS = [row['item_id'] for row in catalog]
ITEM_TO_IDX = {iid: i for i, iid in enumerate(ITEM_IDS)}
CAT_TEXTS = [metadata_text(row) for row in catalog]
del catalog
if len(ITEM_IDS) != len(ITEM_TO_IDX) or len(ITEM_IDS) <= NEGATIVE_COUNT:
    raise ValueError('카탈로그 상품 ID가 중복되거나 서로 다른 오답 상품 수가 부족합니다.')
TOPK = min(CANDIDATE_TOPK, len(ITEM_IDS))
raw_gold = read_json(GOLD_PATH)
BY_QUERY = {row['query_id']: gold_query(row) for row in raw_gold}
if len(BY_QUERY) != len(raw_gold):
    raise ValueError('train_gold.json의 query_id가 중복됩니다.')
del raw_gold

split = read_json(SPLIT_PATH)
SEED = int(split['seed'])
train_ids, val_ids = split['train_query_ids'], split['val_query_ids']
TRAIN = [BY_QUERY[qid] for qid in train_ids]
all_val = [BY_QUERY[qid] for qid in val_ids]
if (
    not TRAIN
    or not all_val
    or len(train_ids) != len(set(train_ids))
    or len(val_ids) != len(set(val_ids))
    or {r['item_id'] for r in TRAIN} & {r['item_id'] for r in all_val}
    or any(r['item_id'] not in ITEM_TO_IDX for r in chain(TRAIN, all_val))
):
    raise ValueError('Gold split의 쿼리 중복, 상품 분리 및 카탈로그 대응을 확인하세요.')
VAL = all_val[:VAL_LIMIT]
del all_val

# 카테고리 후보는 TRAIN 상품으로만 구성한다. 같은 상품의 여러 쿼리가 후보를 중복시키지 않는다.
TRAIN_CATEGORY = {ITEM_TO_IDX[r['item_id']]: r['category_path'] for r in TRAIN if r['category_path']}
CAT_TO_DOCS, PARENT_TO_DOCS = defaultdict(list), defaultdict(list)
for doc, path in TRAIN_CATEGORY.items():
    CAT_TO_DOCS[path].append(doc)
    PARENT_TO_DOCS[path[:-1]].append(doc)

# 인덱스 키는 ColBERT 추가 학습 노트북의 계산식과 같다. 같은 상품 인코더의 기존 인덱스를 찾는다.
precision = 'bfloat16' if USE_BF16 else 'float32'
INDEX_SPEC = {
    'weights': sha256(RETRIEVER_WEIGHTS),
    'catalog': sha256(CATALOG_PATH),
    'seed': SEED,
    'precision': precision,
    'config': {
        k: CFG[k]
        for k in (
            'backbone',
            'dimension',
            'metadata_tokens',
            'ann_nlist',
            'index_doc_batch',
            'index_add_docs',
        )
    },
}
index_key = signature(INDEX_SPEC)[:20]
PREVIOUS_INDEX = RETRIEVER_DIR / 'indexes' / index_key
index_files = ('token_ivf.index', 'token_to_item.npy', 'doc_embeddings.npy', 'doc_masks.npy')
if all((PREVIOUS_INDEX / name).is_file() for name in index_files):
    INDEX_DIR = PREVIOUS_INDEX
else:
    INDEX_DIR = CROSS_ROOT / 'indexes' / index_key
# 이전 epoch 5 후보와 동일한 검색 설정 해시를 만든다. Gold 초기 인덱스를 쓴 기록은 이 키에 해당하지 않는다.
MINING_SPEC = {
    'version': 1,
    'index': INDEX_SPEC,
    'original_index_files': None,
    'gold': sha256(GOLD_PATH),
    'split': sha256(SPLIT_PATH),
    'query_tokens': CFG['query_tokens'],
    'ann_nprobe': CFG['ann_nprobe'],
    'ann_per_token': ANN_PER_TOKEN,
    'ann_pool': ANN_POOL,
    'candidate_topk': CANDIDATE_TOPK,
}
mining_key = signature(MINING_SPEC)[:20]
PREVIOUS_CANDIDATES = PREVIOUS_CACHE_ROOT / mining_key
CANDIDATE_DIR = CROSS_ROOT / 'candidate_cache' / mining_key
TRAINING_SPEC = {
    'version': 1,
    'cross_source': sha256(CROSS_RESUME),
    'mining': signature(MINING_SPEC),
    'seed': SEED,
    'colbert_epoch': COLBERT_EPOCH,
    'base_cross_epoch': BASE_CROSS_EPOCH,
    'target_epochs': list(TARGET_EPOCHS),
    'selection': 'maxsim_top_wrong_plus_gold_sampler_v1',
    'hard_count': HARD_COUNT,
    'original_count': ORIGINAL_COUNT,
    'original_mode': ORIGINAL_MODE,
    'negative_count': NEGATIVE_COUNT,
    'train_sampling': 'epoch_rng_after_shuffle',
    'val_sampling': 'fixed_seed_gold_path_train_category_pools',
    'config': {
        k: CFG[k]
        for k in ('backbone', 'cross_tokens', 'cross_tokens_per_review', 'cross_batch', 'max_grad_norm')
    },
    'lr': CROSS_LR,
    'precision': precision,
}
TRAINING_ID = signature(TRAINING_SPEC)
RUN_OUTPUT = CROSS_ROOT / f'colbert_epoch_{COLBERT_EPOCH:02d}' / TRAINING_ID[:12]
RUN_OUTPUT.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_OUTPUT / 'training_progress.log'
VAL_KEY = signature(
    {'query_ids': [r['query_id'] for r in VAL], 'cross_infer_batch': CFG['cross_infer_batch']}
)[:16]
print(f'고정 검색기 {RETRIEVER_WEIGHTS}\nGold Cross 시작점 {CROSS_RESUME}\n상품 인덱스 {INDEX_DIR}')
print(f'이전 후보 읽기 {PREVIOUS_CANDIDATES}\n새 후보 저장 {CANDIDATE_DIR}')
print(
    f'Cross {BASE_CROSS_EPOCH} → {TARGET_EPOCHS[-1]} | Gold AdamW lr {gold_lrs} → {CROSS_LR:g} | '
    f'검색 오답 {HARD_COUNT} + 기존 {ORIGINAL_MODE} 오답 {ORIGINAL_COUNT} | '
    f'TRAIN {len(TRAIN):,} | VAL {len(VAL):,}\n진행 로그 {LOG_PATH}',
    flush=True,
)

In [ ]:
# 저장한 가중치를 담을 구조와 이전 실행의 토크나이저를 준비한다.
BACKBONE_CONFIG = AutoConfig.from_pretrained(CFG['backbone'])
TOKENIZER = AutoTokenizer.from_pretrained(CFG['backbone'], use_fast=True)
SPECIAL_IDS = set(TOKENIZER.all_special_ids)


# 텍스트 배치를 토큰 ID와 attention mask 텐서로 바꾸고, 길이 제한과 패딩을 적용해 모델의 실행 장치로 보낸다.
def tokenize(texts, max_length):
    out = TOKENIZER(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    return {k: v.to(DEVICE) for k, v in out.items()}


# 재랭커와 baseline은 리뷰 다섯 개를 한 텍스트로 읽는다. 각 리뷰를 자른 뒤 번호와 [SEP]로 경계를 표시한다.
def joined_reviews(reviews):
    # 리뷰별 상한을 먼저 적용한다. 재랭커가 상품과 합쳐 다시 자를 때 뒤쪽 리뷰가 더 짧아질 수 있다.
    chunks = []
    for i, r in enumerate(reviews):
        ids = TOKENIZER.encode(
            r, add_special_tokens=False, truncation=True, max_length=CFG['cross_tokens_per_review']
        )
        chunks.append(f'Review {i + 1}: ' + TOKENIZER.decode(ids, skip_special_tokens=True))
    return ' [SEP] '.join(chunks)


# 후보 검색에 필요한 리뷰·상품 인코더와 토큰 투영층을 구성한다.
# 3. 학습 체크포인트의 구조를 복원한다. 가중치는 .pt에서, 백본 구조와 토크나이저는 학습 설정에서 읽는다.
class ColBERTRetriever(nn.Module):
    def __init__(self, config, dimension):
        super().__init__()
        self.query_encoder = AutoModel.from_config(config)
        self.doc_encoder = AutoModel.from_config(config)
        self.query_proj = nn.Linear(config.hidden_size, dimension, bias=False)
        self.doc_proj = nn.Linear(config.hidden_size, dimension, bias=False)

    # 학습 노트북과 같은 정규화와 유효 토큰 마스크를 사용해 MaxSim용 벡터를 반환한다.
    def encode(self, tokens, side):
        backbone = self.query_encoder if side == 'query' else self.doc_encoder
        projection = self.query_proj if side == 'query' else self.doc_proj
        hidden = backbone(**tokens).last_hidden_state
        mask = tokens['attention_mask'].bool()
        special = torch.zeros_like(mask)
        for token_id in SPECIAL_IDS:
            special |= tokens['input_ids'].eq(token_id)
        valid = mask & ~special
        empty = ~valid.any(dim=-1)
        if empty.any():
            valid[empty] = mask[empty]
        return F.normalize(projection(hidden), p=2, dim=-1), valid


# 검색기가 추린 후보마다 다섯 리뷰와 해당 상품 설명을 함께 읽고, 관련도 점수 하나를 내는 재랭커다.
class CrossReranker(nn.Module):
    # 저장한 Cross 가중치 및 AdamW와 같은 순서로 파라미터를 구성한다.
    def __init__(self):
        super().__init__()
        config = copy.deepcopy(BACKBONE_CONFIG)
        config.num_labels = 1
        self.net = AutoModelForSequenceClassification.from_config(config)

    # 각 리뷰 묶음과 후보 메타데이터를 한 쌍으로 인코딩해, 쌍마다 관련도 점수(logit) 하나를 반환한다.
    def score(self, reviews, doc_texts):
        # 총길이가 cross_tokens를 넘으면 더 긴 쪽부터 줄인다. 이때 뒤쪽 리뷰 전체가 잘릴 수도 있다.
        inputs = TOKENIZER(
            reviews,
            doc_texts,
            padding=True,
            truncation='longest_first',
            max_length=CFG['cross_tokens'],
            return_tensors='pt',
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        return self.net(**inputs).logits.reshape(-1)

## 쿼리당 오답 8개 구성

1. **검색 오답 4개**. ColBERT epoch 5가 토큰당 1,024개 이웃에서 ANN 상품 후보 최대 900개를 찾는다. exact MaxSim 상위 100개에서 Gold 상품을 제외하고 가장 높은 오답 4개를 고른다. 같은 쿼리는 두 학습 에포크에서 이 4개를 공유한다.
2. **기존 방식 오답 4개**. 원래 `mixed` 설정은 전체 카탈로그에서 무작위 2개, 같은 부모·다른 말단 카테고리에서 1개, 같은 말단 경로에서 1개를 고른다. 카테고리 후보가 부족하면 빈자리만 무작위로 채운다. Gold 상품과 앞에서 뽑은 상품은 매번 제외한다. 원래 config가 `random`인 실험에서는 이 4개를 모두 무작위로 뽑는다.

TRAIN에서는 기존 sampler의 난수 시드와 셔플 순서를 따라 에포크마다 두 번째 그룹을 새로 뽑는다. VAL에서는 고정된 시드와 각 쿼리의 Gold 경로를 써서 매 에포크 같은 8개 오답으로 loss를 비교한다. 카테고리별 후보 목록을 만드는 데에는 TRAIN 상품만 쓴다.


In [ ]:
# 카탈로그를 한 번 인코딩해 상품 토큰·마스크와 FAISS IVF를 기록한다.
@torch.inference_mode()
def build_token_index(model, folder):
    folder.mkdir(parents=True)
    n_items, length, dim = len(ITEM_IDS), CFG['metadata_tokens'], CFG['dimension']
    emb = np.lib.format.open_memmap(
        folder / 'doc_embeddings.npy', mode='w+', dtype=np.float16, shape=(n_items, length, dim)
    )
    mask = np.lib.format.open_memmap(
        folder / 'doc_masks.npy', mode='w+', dtype=np.bool_, shape=(n_items, length)
    )
    batch = CFG['index_doc_batch']
    for start in tqdm(range(0, n_items, batch), desc='Encode catalog'):
        end = min(n_items, start + batch)
        tokens = tokenize(CAT_TEXTS[start:end], length)
        with amp_context():
            reps, valid = model.encode(tokens, 'doc')
        width = reps.shape[1]
        emb[start:end] = 0
        mask[start:end] = False
        emb[start:end, :width] = reps.float().cpu().numpy().astype(np.float16)
        mask[start:end, :width] = valid.cpu().numpy()
    emb.flush()
    mask.flush()

    # 학습 노트북과 같은 방식으로 최대 12만 유효 토큰을 골라 IVF의 군집을 구성한다.
    valid_docs, valid_positions = np.where(mask)
    n_train = min(120_000, len(valid_docs))
    sample = np.random.default_rng(SEED).choice(len(valid_docs), size=n_train, replace=False)
    train_vectors = np.ascontiguousarray(emb[valid_docs[sample], valid_positions[sample]], dtype=np.float32)
    token2doc = valid_docs.astype(np.int32)
    del valid_docs, valid_positions
    nlist = min(CFG['ann_nlist'], max(1, n_train // 40))
    index = faiss.IndexIVFFlat(faiss.IndexFlatIP(dim), dim, nlist, faiss.METRIC_INNER_PRODUCT)
    index.train(train_vectors)
    index.nprobe = min(CFG['ann_nprobe'], nlist)
    batch = CFG['index_add_docs']
    for start in tqdm(range(0, n_items, batch), desc='Build token ANN'):
        end = min(n_items, start + batch)
        valid = mask[start:end].reshape(-1)
        vectors = emb[start:end].reshape(-1, dim)[valid]
        if len(vectors):
            index.add(np.ascontiguousarray(vectors, dtype=np.float32))
    assert index.ntotal == len(token2doc)
    faiss.write_index(index, str(folder / 'token_ivf.index'))
    np.save(folder / 'token_to_item.npy', token2doc)


def open_token_index(folder):
    # 같은 인덱스의 네 파일을 읽고 상품 수·토큰 길이·임베딩 차원이 맞는지 확인한다.
    emb = np.load(folder / 'doc_embeddings.npy', mmap_mode='r')
    mask = np.load(folder / 'doc_masks.npy', mmap_mode='r')
    token2doc = np.load(folder / 'token_to_item.npy', mmap_mode='r')
    index = faiss.read_index(str(folder / 'token_ivf.index'))
    expected = (len(ITEM_IDS), CFG['metadata_tokens'], CFG['dimension'])
    if (
        emb.shape != expected
        or mask.shape != expected[:2]
        or mask.dtype != np.bool_
        or index.d != expected[2]
        or token2doc.ndim != 1
        or index.ntotal != len(token2doc)
    ):
        raise ValueError(f'상품 인덱스의 크기가 현재 설정과 맞지 않습니다. {folder}')
    if not len(token2doc) or token2doc.min() < 0 or token2doc.max() >= len(ITEM_IDS):
        raise ValueError(f'인덱스의 토큰-상품 대응표를 확인하세요. {folder}')
    index.nprobe = min(CFG['ann_nprobe'], index.nlist)
    return {'index': index, 'token2doc': token2doc, 'emb': emb, 'mask': mask}


def publish_directory(staged, destination):
    # 같은 파일시스템에서 완성된 폴더만 공개한다. 동시 실행에서 먼저 완성된 캐시를 공유한다.
    try:
        staged.rename(destination)
    except OSError:
        if not destination.is_dir():
            raise


# 쿼리 한 건의 다섯 리뷰를 각각 토큰 임베딩으로 바꾼다.
# 5. 검색 점수는 원본과 같은 방식으로 계산하고, 각 중간 순위와 정답 점수를 함께 남긴다.
@torch.inference_mode()
def encode_five_reviews(model, reviews):
    tokens = tokenize(reviews, CFG['query_tokens'])
    with amp_context():
        embeddings, valid = model.encode(tokens, 'query')
    return embeddings.float().cpu().numpy(), valid.cpu().numpy()


@torch.inference_mode()
def maxsim_scores(query, query_mask, index_bundle, doc_indices):
    # 후보의 유효 상품 토큰 전체를 읽고, 리뷰 토큰마다 가장 높은 내적을 더한다.
    # 최종 점수 하나는 다섯 리뷰의 MaxSim 점수 평균이다.
    qe = torch.as_tensor(query, device=DEVICE)
    qm = torch.as_tensor(query_mask, device=DEVICE)
    scores = []
    for start in range(0, len(doc_indices), 96):
        rows = doc_indices[start : start + 96]
        de = torch.as_tensor(np.asarray(index_bundle['emb'][rows]).astype(np.float32), device=DEVICE)
        dm = torch.as_tensor(np.asarray(index_bundle['mask'][rows]), device=DEVICE)
        with amp_context():
            simil = torch.einsum('rqd,cmd->rcqm', qe, de)
            simil = simil.masked_fill(~dm[None, :, None, :], -1e4)
            review_scores = (simil.max(-1).values * qm[:, None, :]).sum(-1)
        scores.append(review_scores.mean(0).float().cpu().numpy())
    return np.concatenate(scores)


def scored_pool(ann_order, pool_size, topk):
    # ANN 적중 여부는 보충 전 순위로 센다. 실제 후보 부족 시에만 원래 검색기와 같은 순서로 보충한다.
    pool = ann_order[:pool_size]
    if len(pool) < topk:
        seen = set(pool)
        need = min(pool_size, len(ITEM_IDS)) - len(pool)
        pool.extend(islice((i for i in range(len(ITEM_IDS)) if i not in seen), need))
    return pool

In [ ]:
def frozen_retriever():
    # 가중치 파일에서 리뷰·상품 인코더와 토큰 투영층을 복원한다. 이 모델은 후보 생성에만 쓴다.
    model = ColBERTRetriever(BACKBONE_CONFIG, CFG['dimension'])
    state = load_pt(RETRIEVER_WEIGHTS)
    model.load_state_dict({k: state[k] for k in model.state_dict()}, strict=True)
    return model.to(DEVICE).eval().requires_grad_(False)


def open_mining_index(model):
    # 같은 체크포인트의 인덱스를 공유한다. 처음 쓸 때는 완성된 폴더를 한 번에 설치한다.
    if INDEX_DIR == PREVIOUS_INDEX:
        print('기존 epoch 5 상품 인덱스 읽기', INDEX_DIR, flush=True)
        return open_token_index(INDEX_DIR)
    if not INDEX_DIR.is_dir():
        INDEX_DIR.parent.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(dir=INDEX_DIR.parent) as tmp:
            staged = Path(tmp) / 'index'
            print('상품 인덱스 생성 중', INDEX_DIR, flush=True)
            build_token_index(model, staged)
            write_json(staged / 'index_info.json', INDEX_SPEC)
            publish_directory(staged, INDEX_DIR)
    print('사용할 상품 인덱스', INDEX_DIR, flush=True)
    return open_token_index(INDEX_DIR)


def ann_order(query, mask, bundle):
    # 같은 (리뷰 토큰, 상품)에서 최대 유사도 하나를 고른 뒤 양수 부분을 상품별로 합산한다.
    # 정렬·집계를 NumPy로 처리해 TRAIN 전체에서 토큰 이웃을 파이썬 반복문으로 순회하는 비용을 줄인다.
    tokens = np.ascontiguousarray(query[mask], dtype=np.float32)
    distances, hits = bundle['index'].search(tokens, ANN_PER_TOKEN)
    rows, columns = np.nonzero(hits >= 0)
    if rows.size == 0:
        return []
    docs = bundle['token2doc'][hits[rows, columns]]
    keys = rows.astype(np.int64) * len(ITEM_IDS) + docs
    order = np.argsort(keys, kind='stable')
    keys = keys[order]
    starts = np.r_[0, np.flatnonzero(keys[1:] != keys[:-1]) + 1]
    maxima = np.maximum.reduceat(distances[rows, columns][order], starts)
    docs = keys[starts] % len(ITEM_IDS)
    rough = np.bincount(docs, weights=np.maximum(maxima, 0.0), minlength=len(ITEM_IDS))
    touched = np.unique(docs)
    return touched[np.argsort(-rough[touched], kind='stable')[:ANN_POOL]].tolist()


@torch.inference_mode()
def retrieve_candidates(row, retriever, bundle):
    # ANN 후보를 exact MaxSim으로 다시 정렬한다. 원본처럼 top-K보다 적게 찾은 때만 풀을 보충한다.
    query, mask = encode_five_reviews(retriever, row['reviews'])
    ann = ann_order(query, mask, bundle)
    gold = ITEM_TO_IDX[row['item_id']]
    ann_rank = ann.index(gold) + 1 if gold in ann else 0
    pool = scored_pool(ann, ANN_POOL, TOPK)
    scores = maxsim_scores(query, mask, bundle, pool)
    indices = np.argsort(-scores, kind='stable')[:TOPK]
    return np.asarray(pool, dtype=np.int32)[indices], ann_rank


def candidate_tables(log):
    # 쿼리 순서와 가중치·검색 설정을 기준으로 캐시를 나눈다. TRAIN과 VAL은 별도 파일을 쓴다.
    # 중단돼도 완전히 저장된 쿼리 묶음은 재검색하지 않는다. 한 행에는 MaxSim 순서대로 상품 행 번호가 들어간다.
    retriever = bundle = None
    tables = {}
    for name, rows in [('train', TRAIN), ('val', VAL)]:
        qids = [r['query_id'] for r in rows]
        split_key = f'{name}_{signature(qids)[:16]}'
        folder = CANDIDATE_DIR / split_key
        previous_folder = PREVIOUS_CANDIDATES / split_key
        candidates = np.empty((len(rows), TOPK), dtype=np.int32)
        ann_ranks = np.empty(len(rows), dtype=np.int32)
        cached_count = previous_count = 0
        log.write(f'mining {name} | preparing {len(rows)} queries | {folder}\n')
        for start in tqdm(range(0, len(rows), MINING_CHUNK_SIZE), desc=f'{name} 후보 준비'):
            end = min(start + MINING_CHUNK_SIZE, len(rows))
            path = folder / f'{start:06d}_{end:06d}.npz'
            expected_ids = np.asarray(qids[start:end], dtype=str)
            # 이전 실행에서 완료한 묶음은 그대로 읽는다. 새 묶음은 이 실험 전용 캐시에만 쓴다.
            cached = path if path.is_file() else previous_folder / path.name
            if cached.is_file():
                with np.load(cached, allow_pickle=False) as shard:
                    if not np.array_equal(shard['query_ids'], expected_ids):
                        raise ValueError(f'저장한 후보의 query_id 순서를 확인하세요. {cached}')
                    saved_candidates, saved_ranks = shard['candidates'], shard['ann_ranks']
                    if saved_candidates.shape != (end - start, TOPK) or saved_ranks.shape != (end - start,):
                        raise ValueError(f'저장한 후보 배열의 크기를 확인하세요. {cached}')
                    candidates[start:end] = saved_candidates
                    ann_ranks[start:end] = saved_ranks
                    cached_count += end - start
                    if cached != path:
                        previous_count += end - start
            else:
                if retriever is None:
                    log.write(f'mining | loading ColBERT | index {INDEX_DIR}\n')
                    retriever = frozen_retriever()
                    bundle = open_mining_index(retriever)
                for i in range(start, end):
                    candidates[i], ann_ranks[i] = retrieve_candidates(rows[i], retriever, bundle)
                folder.mkdir(parents=True, exist_ok=True)
                with TemporaryDirectory(dir=folder) as tmp:
                    staged = Path(tmp) / path.name
                    np.savez(
                        staged,
                        query_ids=expected_ids,
                        candidates=candidates[start:end],
                        ann_ranks=ann_ranks[start:end],
                    )
                    staged.replace(path)
                log.write(f'mining {name} | {end}/{len(rows)} queries saved\n')
        if (
            candidates.min() < 0
            or candidates.max() >= len(ITEM_IDS)
            or np.any(np.diff(np.sort(candidates, axis=1), axis=1) == 0)
        ):
            raise ValueError(f'중복 또는 범위를 벗어난 후보 상품이 있습니다. {folder}')
        tables[name] = (candidates, ann_ranks)
        log.write(
            f'mining {name} | ready {len(rows)} | reused {cached_count} cached queries'
            f' ({previous_count} from previous experiment)\n'
        )
        print(f'{name} 후보 {len(rows):,}건 준비 완료 · 기존 실험 캐시 {previous_count:,}건 사용', flush=True)
    return tables


def sample_original_negatives(gold, hard, category, rng):
    # 기존 Gold sampler의 추출 순서를 유지하되, 검색 오답 4개도 처음부터 제외한다.
    picked = []
    excluded = {gold, *hard}

    def fill_random(target):
        # 전체 카탈로그에서 현재 목록이 target 길이가 될 때까지 중복 없이 뽑는다.
        while len(picked) < target:
            doc = rng.randrange(len(ITEM_IDS))
            if doc not in excluded:
                picked.append(doc)
                excluded.add(doc)

    def add_category(pool, count):
        # 정답과 앞서 뽑은 오답을 제외한 뒤, 카테고리에 남아 있는 만큼만 뽑는다.
        available = [doc for doc in pool if doc not in excluded]
        chosen = rng.sample(available, min(count, len(available)))
        picked.extend(chosen)
        excluded.update(chosen)

    if ORIGINAL_MODE == 'mixed' and category:
        random_count = ORIGINAL_COUNT // 2
        sibling_count = (ORIGINAL_COUNT - random_count) // 2
        fill_random(random_count)
        siblings = (doc for doc in PARENT_TO_DOCS.get(category[:-1], ()) if TRAIN_CATEGORY[doc] != category)
        add_category(siblings, sibling_count)
        add_category(CAT_TO_DOCS.get(category, ()), ORIGINAL_COUNT - random_count - sibling_count)
    fill_random(ORIGINAL_COUNT)
    return picked


def mixed_groups(rows, candidates, rng):
    # 행마다 [Gold, MaxSim 상위 오답 4개, 기존 방식 오답 4개]를 만들어 Cross의 정답 열을 0번으로 맞춘다.
    groups = np.empty((len(rows), 1 + NEGATIVE_COUNT), dtype=np.int32)
    for i, row in enumerate(rows):
        gold = ITEM_TO_IDX[row['item_id']]
        ranked = candidates[i]
        hard = ranked[ranked != gold][:HARD_COUNT].tolist()
        category = TRAIN_CATEGORY.get(gold, row['category_path'])
        original = sample_original_negatives(gold, hard, category, rng)
        groups[i] = [gold, *hard, *original]
    return groups

## Cross 학습과 체크포인트

Cross는 리뷰 다섯 개와 각 상품의 메타데이터를 함께 읽어 쿼리당 9개 점수를 낸다. 정답은 0번 열이며 기존 listwise cross-entropy를 적용한다. Gold epoch 2의 AdamW step과 모멘트를 복원한 **후에** 모든 파라미터 그룹의 학습률을 **5e-6**으로 설정한다.

에포크마다 검증에 들어가기 전에 추론용 `cross_mixed.pt`와 재개용 `cross_mixed_last.pt`를 함께 저장한다. 재개 파일에는 모델, optimizer, 난수 상태, 평균 train loss, 실험 ID가 들어간다. 재실행 시 이번 실험 경로·ID·에포크와 일치하는 재개 파일만 사용한다.


In [ ]:
@lru_cache(maxsize=None)
def cross_query_text(query_id):
    # 같은 다섯 리뷰로 여러 상품을 채점한다. 원본과 같은 리뷰별 토큰 제한과 구분자를 한 번만 적용한다.
    return joined_reviews(BY_QUERY[query_id]['reviews'])


def group_loss(logits):
    # 각 행은 정답 1개와 두 방식으로 뽑은 오답 8개다. 원래 Cross 랭킹 손실과 같이 raw logit에 CE를 적용한다.
    labels = torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)
    return F.cross_entropy(logits.float(), labels)


def train_cross_epoch(model, optimizer, number, candidates, log):
    # Gold 학습과 같은 시드로 쿼리를 섞고, 남은 난수 스트림으로 이 에포크의 기존 방식 오답을 뽑는다.
    order = list(range(len(TRAIN)))
    rng = random.Random(SEED + number - 1 + 233)
    rng.shuffle(order)
    batch_size = CFG['cross_batch']
    starts = range(0, len(order), batch_size)
    model.train()
    total_loss = 0.0
    log.write(
        f'{datetime.now().astimezone().isoformat(timespec="seconds")} | cross epoch {number} | started | {len(starts)} batches\n'
    )
    for batch_no, start in enumerate(tqdm(starts, desc=f'Cross gold epoch {number}'), start=1):
        indices = order[start : start + batch_size]
        batch = [TRAIN[i] for i in indices]
        docs = mixed_groups(batch, candidates[indices], rng)
        texts = [cross_query_text(row['query_id']) for row in batch]
        reviews = [text for text in texts for _ in range(docs.shape[1])]
        metadata = [CAT_TEXTS[i] for i in docs.ravel()]
        optimizer.zero_grad(set_to_none=True)
        with amp_context():
            scores = model.score(reviews, metadata).reshape(len(indices), docs.shape[1])
            loss = group_loss(scores)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG['max_grad_norm'])
        optimizer.step()
        batch_loss = loss.detach().item()
        total_loss += batch_loss * len(indices)
        log.write(
            f'cross epoch {number} | batch {batch_no}/{len(starts)} | completed | loss {batch_loss:.6f}\n'
        )
    return total_loss / len(TRAIN)


def restore_cross(model, optimizer, path, number):
    # 최초 입력은 원본 Gold epoch 2, 이후 입력은 이 실험 폴더의 재개 파일로 제한한다.
    from_gold = number == BASE_CROSS_EPOCH
    expected = (
        CROSS_RESUME if from_gold else RUN_OUTPUT / f'epoch_{number:02d}' / 'checkpoints' / MIXED_RESUME_NAME
    )
    if path != expected or (not from_gold and number not in TARGET_EPOCHS):
        raise ValueError(f'이 실험의 Cross 재개 경로와 다릅니다. {path}')
    state = load_pt(path)
    if int(state['epoch']) != number or not state['optimizer_state_dict']['state']:
        raise ValueError(f'Cross epoch 또는 AdamW 상태가 맞지 않습니다. {path}')
    saved_groups = state['optimizer_state_dict']['param_groups']
    if not from_gold and (
        state.get('training_id') != TRAINING_ID or any(float(g['lr']) != CROSS_LR for g in saved_groups)
    ):
        raise ValueError(f'실험 ID 또는 학습률이 다른 Cross 체크포인트입니다. {path}')
    model.load_state_dict(state['model_state_dict'], strict=True)
    optimizer.load_state_dict(state['optimizer_state_dict'])
    # load_state_dict가 Gold의 기존 lr도 불러오므로 step과 모멘트 복원을 마친 뒤 새 값을 적용한다.
    if from_gold:
        for group in optimizer.param_groups:
            group['lr'] = CROSS_LR
    # 원래 Gold 파일에는 dropout 난수 상태가 없다. 새 에포크 파일부터 저장된 상태를 이어 쓴다.
    torch.manual_seed(SEED + number)
    if 'torch_rng_state' in state:
        torch.set_rng_state(state['torch_rng_state'])
    if DEVICE.type == 'cuda' and state.get('cuda_rng_state') is not None:
        torch.cuda.set_rng_state(state['cuda_rng_state'], device=DEVICE)
    print(f'Cross epoch {number} 복원 | AdamW lr {optimizer.param_groups[0]["lr"]:g} | {path}', flush=True)
    return state.get('mean_train_loss')


def save_cross_epoch(model, optimizer, number, train_loss):
    # 검증 전에 재개용 전체 상태와 추론용 가중치를 함께 저장한다. 완성된 폴더만 외부에 공개한다.
    parent = RUN_OUTPUT / f'epoch_{number:02d}'
    parent.mkdir(parents=True, exist_ok=True)
    folder = parent / 'checkpoints'
    weights = model.state_dict()
    state = {
        'epoch': number,
        'additional_epoch': number - BASE_CROSS_EPOCH,
        'model_state_dict': weights,
        'optimizer_state_dict': optimizer.state_dict(),
        'mean_train_loss': train_loss,
        'training_id': TRAINING_ID,
        'source_checkpoint': str(CROSS_RESUME),
        'retriever_checkpoint': str(RETRIEVER_WEIGHTS),
        'hard_count': HARD_COUNT,
        'original_count': ORIGINAL_COUNT,
        'original_mode': ORIGINAL_MODE,
        'lr': optimizer.param_groups[0]['lr'],
        'seed': SEED,
        'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state': torch.cuda.get_rng_state(DEVICE) if DEVICE.type == 'cuda' else None,
    }
    with TemporaryDirectory(dir=parent) as tmp:
        staged = Path(tmp) / 'checkpoints'
        staged.mkdir()
        torch.save(state, staged / MIXED_RESUME_NAME)
        torch.save(weights, staged / MIXED_WEIGHTS_NAME)
        staged.rename(folder)
    print('Cross 체크포인트 저장', folder, flush=True)

## 같은 VAL 후보로 epoch 2·3·4 비교

VAL loss는 정답 1개, 검색 오답 4개, 기존 방식 오답 4개의 고정 그룹에서 계산한다. 카테고리·무작위 오답이나 Gold가 검색 후보 밖에 있으면 loss 계산을 위해 추가로 점수를 구한다. 검색 지표는 처음 검색한 100개 상품의 순위로만 계산한다.

`ann_recall_at_900`과 `colbert_recall_at_100`은 고정 검색기를 확인하는 지표다. `cross_precision_at_1`과 `cross_recall_at_10`은 Cross 가중치가 바뀐 효과를 보여준다. 기준점인 epoch 2는 검증만 수행하므로 train loss 칸이 비어 있다.


In [ ]:
@torch.inference_mode()
def validate_cross(model, candidates, ann_ranks, groups, number):
    # Precision·Recall은 실제 검색 후보만으로 계산한다. VAL loss는 TRAIN과 같이 정답 1개와 고정 오답 N개로 잰다.
    model.eval()
    records = []
    for i, row in enumerate(tqdm(VAL, desc=f'Cross VAL epoch {number}')):
        ranked = candidates[i].tolist()
        gold = int(groups[i, 0])
        hit = gold in ranked
        # 검색 밖의 Gold·기존 방식 오답은 VAL loss에 필요하다. 뒤에 한 번씩 붙여 원래 순위 구간을 보존한다.
        scored = list(dict.fromkeys([*ranked, *groups[i].tolist()]))
        query = cross_query_text(row['query_id'])
        logits = []
        for start in range(0, len(scored), CFG['cross_infer_batch']):
            docs = scored[start : start + CFG['cross_infer_batch']]
            with amp_context():
                out = model.score([query] * len(docs), [CAT_TEXTS[j] for j in docs])
            logits.append(out.float().cpu())
        logits = torch.cat(logits)
        positions = {doc: j for j, doc in enumerate(scored)}
        group_positions = [positions[int(doc)] for doc in groups[i]]
        val_loss = group_loss(logits[group_positions].unsqueeze(0)).item()
        order = np.argsort(-logits[: len(ranked)].numpy(), kind='stable')
        final = [ranked[j] for j in order]
        cross_rank = final.index(gold) + 1 if hit else None
        records.append(
            {
                'query_id': row['query_id'],
                'gold_item_id': row['item_id'],
                'val_loss': val_loss,
                'gold_ann_rank_in_pool': int(ann_ranks[i]) or None,
                'gold_maxsim_rank_in_topk': ranked.index(gold) + 1 if hit else None,
                'cross_gold_rank': cross_rank,
                'cross_top1_item_id': ITEM_IDS[final[0]],
                'ann_hit': bool(ann_ranks[i]),
                'colbert_hit': hit,
                'cross_top1_hit': cross_rank == 1,
                'cross_top10_hit': cross_rank is not None and cross_rank <= 10,
            }
        )
    return pd.DataFrame.from_records(records)


def record_validation(model, candidates, ann_ranks, groups, number, train_loss, weights_path, log):
    # 같은 Cross 가중치와 VAL 후보로 끝낸 평가는 다시 읽는다. 검증 중 끊긴 경우에는 저장한 가중치로 재평가한다.
    eval_id = signature({'cross': sha256(weights_path), 'val': VAL_KEY, 'version': 1})[:16]
    parent = RUN_OUTPUT / f'epoch_{number:02d}' / 'validation'
    report = parent / eval_id
    if not (report / 'stage_summary.csv').is_file():
        log.write(f'cross epoch {number} | validation started | {len(VAL)} queries\n')
        detail = validate_cross(model, candidates, ann_ranks, groups, number)
        detail.insert(0, 'epoch', number)
        candidate_hits = int(detail['colbert_hit'].sum())
        summary = {
            'epoch': number,
            'additional_epoch': number - BASE_CROSS_EPOCH,
            'val_set': VAL_KEY,
            'colbert_epoch': COLBERT_EPOCH,
            'ann_per_token': ANN_PER_TOKEN,
            'ann_pool': ANN_POOL,
            'candidate_topk': TOPK,
            'negative_count': NEGATIVE_COUNT,
            'hard_count': HARD_COUNT,
            'original_count': ORIGINAL_COUNT,
            'original_mode': ORIGINAL_MODE,
            'train_lr': CROSS_LR if train_loss is not None else None,
            'n_train': len(TRAIN) if train_loss is not None else 0,
            'n_val': len(VAL),
            'n_queries': len(VAL),
            'train_loss': train_loss,
            'val_loss': detail['val_loss'].mean(),
            'ann_hits': int(detail['ann_hit'].sum()),
            'colbert_hits': candidate_hits,
            'top1_hits': int(detail['cross_top1_hit'].sum()),
            'ann_recall_at_pool': detail['ann_hit'].mean(),
            'colbert_recall_at_k': detail['colbert_hit'].mean(),
            'cross_precision_at_1': detail['cross_top1_hit'].mean(),
            'cross_recall_at_10': detail['cross_top10_hit'].mean(),
            'cross_top1_given_candidate': detail['cross_top1_hit'].sum() / candidate_hits
            if candidate_hits
            else None,
        }
        if ANN_POOL == 900:
            summary['ann_recall_at_900'] = summary['ann_recall_at_pool']
        if TOPK == 100:
            summary['colbert_recall_at_100'] = summary['colbert_recall_at_k']
        parent.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(dir=parent) as tmp:
            staged = Path(tmp) / 'report'
            staged.mkdir()
            detail.to_csv(staged / 'query_diagnostics.csv', index=False)
            pd.DataFrame([summary]).to_csv(staged / 'stage_summary.csv', index=False)
            publish_directory(staged, report)

    current = pd.read_csv(report / 'stage_summary.csv', dtype={'val_set': str})
    # 에포크가 끝날 때마다 손실과 검색 지표를 한 표에 반영한다. 재실행은 같은 VAL·epoch의 한 행을 갱신한다.
    path = RUN_OUTPUT / 'epoch_losses.csv'
    history = current
    if path.is_file():
        prior = pd.read_csv(path, dtype={'val_set': str})
        keep = ~(prior['epoch'].eq(number) & prior['val_set'].eq(VAL_KEY))
        history = pd.concat([prior.loc[keep], current], ignore_index=True)
    with TemporaryDirectory(dir=RUN_OUTPUT) as tmp:
        staged = Path(tmp) / path.name
        history.sort_values(['val_set', 'epoch']).to_csv(staged, index=False)
        staged.replace(path)
    row = current.iloc[0]
    train_value = f'{train_loss:.4f}' if train_loss is not None else 'Gold 기준점'
    message = (
        f'cross epoch {number} | train {train_value} | val {row.val_loss:.4f} | '
        f'P@1 {row.cross_precision_at_1:.4f} | R@10 {row.cross_recall_at_10:.4f}'
    )
    print(message, flush=True)
    log.write(message + '\n')
    return current

## 실행과 로그

`data304_outputs/cross_hard4_mixed4_lr5e-6/colbert_epoch_05/<실험 ID>/` 아래에 결과가 생긴다. 같은 설정으로 재실행하면 완료된 에포크는 읽고, 검증만 남은 경우 저장한 가중치로 평가한다.

| 경로 | 확인할 내용 |
|---|---|
| `settings.json` | 입력 체크포인트, 검색 조건, sampler, 학습률 |
| `training_progress.log` | 캐시 재사용·검색 진행, 배치별 train loss, 에포크별 train·val loss |
| `epoch_losses.csv` | 기준점과 추가 에포크별 loss 및 순위 지표 |
| `epoch_03/checkpoints/`, `epoch_04/checkpoints/` | `cross_mixed.pt`와 `cross_mixed_last.pt` |
| `epoch_XX/validation/<평가 ID>/` | `stage_summary.csv`와 쿼리별 `query_diagnostics.csv` |

SageMaker의 별도 터미널에서 다음 명령으로 배치 로그를 볼 수 있다.

```bash
tail -n 30 -F /home/ec2-user/SageMaker/20262R0136DATA30400/data304_outputs/cross_hard4_mixed4_lr5e-6/colbert_epoch_05/*/training_progress.log
```


In [ ]:
def continue_cross():
    # Linux/SageMaker에서는 같은 실험을 두 커널에서 동시에 실행해 같은 체크포인트를 쓰는 일을 막는다.
    with LOG_PATH.open('a', encoding='utf-8', buffering=1) as log:
        if os.name == 'posix':
            import fcntl

            try:
                fcntl.flock(log, fcntl.LOCK_EX | fcntl.LOCK_NB)
            except BlockingIOError as exc:
                raise RuntimeError(f'이 실험을 실행 중인 다른 커널이 있습니다. {LOG_PATH}') from exc
        log.write(
            f'{datetime.now().astimezone().isoformat(timespec="seconds")} | run started | ColBERT {COLBERT_EPOCH} | '
            f'Cross {BASE_CROSS_EPOCH} -> {TARGET_EPOCHS[-1]} | lr {CROSS_LR:g} | '
            f'hard {HARD_COUNT} + {ORIGINAL_MODE} {ORIGINAL_COUNT}\n'
        )
        write_json(
            RUN_OUTPUT / 'settings.json',
            {
                'colbert_checkpoint': str(RETRIEVER_WEIGHTS),
                'cross_resume': str(CROSS_RESUME),
                'index_path': str(INDEX_DIR),
                'split': str(SPLIT_PATH),
                'candidate_cache': str(CANDIDATE_DIR),
                'previous_readonly_candidates': str(PREVIOUS_CANDIDATES),
                'gold_optimizer_lrs': gold_lrs,
                'mining': MINING_SPEC,
                'training': TRAINING_SPEC,
                'target_epochs': list(TARGET_EPOCHS),
                'val_query_ids': [row['query_id'] for row in VAL],
                'val_set': VAL_KEY,
            },
        )
        tables = candidate_tables(log)
        train_candidates = tables['train'][0]
        val_candidates, val_ann_ranks = tables['val']
        # VAL 오답은 한 번 뽑아 기준점과 두 추가 에포크의 분모를 같게 유지한다.
        val_groups = mixed_groups(VAL, val_candidates, random.Random(SEED))

        # mining에는 고정 ColBERT를 썼다. 아래 AdamW에는 Cross-Encoder의 파라미터만 전달한다.
        cross = CrossReranker().to(DEVICE)
        optimizer = torch.optim.AdamW(cross.parameters(), lr=CROSS_LR, weight_decay=0.01)
        restore_cross(cross, optimizer, CROSS_RESUME, BASE_CROSS_EPOCH)
        parts = [
            record_validation(
                cross, val_candidates, val_ann_ranks, val_groups, BASE_CROSS_EPOCH, None, CROSS_RESUME, log
            )
        ]
        for number in TARGET_EPOCHS:
            folder = RUN_OUTPUT / f'epoch_{number:02d}' / 'checkpoints'
            resume = folder / MIXED_RESUME_NAME
            weights = folder / MIXED_WEIGHTS_NAME
            if folder.is_dir():
                if not (resume.is_file() and weights.is_file()):
                    raise FileNotFoundError(
                        f'이 에포크의 재개·추론용 Cross 체크포인트를 확인하세요. {folder}'
                    )
                train_loss = restore_cross(cross, optimizer, resume, number)
                log.write(f'cross epoch {number} | loaded completed checkpoint\n')
            else:
                train_loss = train_cross_epoch(cross, optimizer, number, train_candidates, log)
                save_cross_epoch(cross, optimizer, number, train_loss)
                log.write(f'cross epoch {number} | checkpoint saved | {resume}\n')
            parts.append(
                record_validation(
                    cross, val_candidates, val_ann_ranks, val_groups, number, train_loss, weights, log
                )
            )

    result = pd.concat(parts, ignore_index=True)
    display(
        result[
            [
                'epoch',
                'train_loss',
                'val_loss',
                'n_queries',
                'ann_recall_at_pool',
                'colbert_recall_at_k',
                'cross_precision_at_1',
                'cross_recall_at_10',
                'cross_top1_given_candidate',
            ]
        ].round(4)
    )
    print('에포크 비교', RUN_OUTPUT / 'epoch_losses.csv')
    return result


results = continue_cross()